# Set up 
## Compute
- 17.3 LTS Scala 2.13 Spark 4.0.0
- Policy Personal Compute 
## Libraries
- org.neo4j:neo4j-connector-apache-spark_2.13:6.0.0_for_spark_4

For Shared Compute add org.neo4j:neo4j-connector-apache-spark_2.13:6.0.0_for_spark_4 to allow list


In [0]:
# Secrets should not be here 
USER_NAME="neo4j"
PASSWORD="will make example with sso in the future"
DATABASE_NAME="neo4j"
DBMS="neo4j+s://[database instance id goes here].databases.neo4j.io"

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .config("neo4j.url", DBMS)
    .config("neo4j.authentication.basic.username", USER_NAME)
    .config("neo4j.authentication.basic.password", PASSWORD)
    .config("neo4j.database", DATABASE_NAME)
    .getOrCreate()
)

## Patient nodes

Select data, shape it so it fits the nodes you want to create

In [0]:
patients = spark.sql(
    """
    select 
        Id as id,
        BIRTHDATE as birthdate,
        SSN as ssn,
        FIRST as first_name,
        LAST as last_name
    from haklof.default.patients
    """
)
patients.display()

In [0]:
(
    patients.write.format("org.neo4j.spark.DataSource")
    .mode("Overwrite")
    .option("labels", ":Patient")
    .option("node.keys","id")
    .option("schema.optimization.node.keys", "KEY")
    .save()
)

## Encounter nodes

In [0]:
encounters = spark.sql(
    """
    select 
        Id as id,
        START as start,
        STOP as stop,
        ENCOUNTERCLASS as class,
        CODE as code,
        DESCRIPTION as description,
        BASE_ENCOUNTER_COST as base_cost,
        TOTAL_CLAIM_COST as total_claim_cost,
        PAYER_COVERAGE as payer_coverage
    from haklof.default.encounters
    """
)
encounters.display()

In [0]:
(
    encounters.write.format("org.neo4j.spark.DataSource")
    .mode("Overwrite")
    .option("labels", ":Encounter")
    .option("node.keys","id")
    .option("schema.optimization.node.keys", "KEY")
    .save()
)

## Create relationship (Patient) -[:HAS_ENCOUNTER]-> (:Encounter)

For relationships, name the from node - to node keys in a clever way. Select desired additional relationship properties and name them well.

In [0]:
patient_encounters = spark.sql(
    """
    select 
        Id as encounter_id,
        PATIENT as patient_id
    from haklof.default.encounters
    """
)
patient_encounters.display()

In [0]:
(
    patient_encounters.repartition(1).write.mode("Overwrite").format("org.neo4j.spark.DataSource")
    .option("relationship", "HAS_ENCOUNTER")
    .option("relationship.save.strategy", "keys")
    .option("relationship.source.save.mode", "Match")
    .option("relationship.source.labels", ":Patient")
    .option("relationship.source.node.keys", "patient_id:id") # patient_id:id  [column in the dataframe]:[property used as nodekey]
    .option("relationship.target.save.mode", "Match")
    .option("relationship.target.labels", ":Encounter")
    .option("relationship.target.node.keys", "encounter_id:id")
    .save()
)

## Allergies

In [0]:
patient_allergies = spark.sql(
    """
    select 
        CODE as code,
        SYSTEM as system,
        PATIENT as patient_id,
        DESCRIPTION as description,
        TYPE as type,
        CATEGORY as category
    from haklof.default.allergies
    """
)
patient_allergies.display()

In [0]:
write_allergy_query="""
  merge (a:Allergy{code: event.code, system: event.system})
  on create set a.description = event.description,
                a.type = event.type,
                a.category = event.category
  merge (p:Patient{id:event.patient_id})
  merge (p)-[:HAS_ALLERGY]->(a)
"""
(
  patient_allergies.write
  .format("org.neo4j.spark.DataSource")
  .option("query", write_allergy_query)
  .option("script","CREATE CONSTRAINT IF NOT EXISTS FOR (n:Allergy) REQUIRE (n.code, n.system) IS NODE KEY;")\
  .mode("Overwrite")
  .save()
)

In [0]:
patient_allergies_vectors = spark.sql(
    """
    select 
        CODE as code,
        SYSTEM as system,
        `__db_DESCRIPTION_vector` as description_vector
    from haklof.default.allergies_vectors_writeback_table
    """
)
patient_allergies_vectors.display()

In [0]:
write_allergy_vectors_query = """
  merge (a:Allergy{code: event.code, system: event.system})
  set a.description_vector = vector(
      event.description_vector,
      1536,
      FLOAT
    )
"""

(
  patient_allergies_vectors.write
  .format("org.neo4j.spark.DataSource")
  .option("query", write_allergy_vectors_query)
  .option("script","CREATE VECTOR INDEX allergy_description IF NOT EXISTS FOR (n:Allergy) ON (n.description_vector) with [n.system];")\
  .mode("Overwrite")
  .save()
)